In [2]:
import subprocess
import sys
subprocess.check_call([sys.executable,'-m','pip','install',
    'qiskit==2.2.3','qiskit-machine-learning==0.9.0','numpy==2.2.6',
    'scipy==1.15.3','scikit-learn==1.6.1','pandas==2.2.3',
    'matplotlib==3.10.1','flask==3.1.0'])

0

In [3]:
# 2. 데이터에서 예측까지의 흐름
import numpy as np
from sklearn.preprocessing import MinMaxScaler
train=np.array([[100.],[200.],[300.]])
test=np.array([[250.],[400.]])
scaler=MinMaxScaler(clip=True).fit(train)
print('학습 최솟값/최댓값:',scaler.data_min_.tolist(),scaler.data_max_.tolist())
print('시험 변환:',scaler.transform(test).ravel().tolist())

학습 최솟값/최댓값: [100.0] [300.0]
시험 변환: [0.75, 1.0]


In [4]:
# 3. 데이터 인코딩과 상태 변화
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
x=0.3
theta=0.4  # 세타: 학습할 Y축 회전 각도(라디안)
qc=QuantumCircuit(1)
qc.h(0)
qc.p(2*x,0)
before=Statevector.from_instruction(qc)
qc.ry(theta,0)
after=Statevector.from_instruction(qc)
expected=(1+np.sin(theta)*np.cos(2*x))/2
print('인코딩 진폭:',np.round(before.data,6))
print('인코딩 확률:',np.round(before.probabilities(),6))
print('학습 회전 후 진폭:',np.round(after.data,6))
print('P(1):',round(float(after.probabilities()[1]),6))
assert np.isclose(after.probabilities()[1],expected)

인코딩 진폭: [0.707107+0.j       0.5836  +0.399263j]
인코딩 확률: [0.5 0.5]
학습 회전 후 진폭: [0.577068-0.079321j 0.712448+0.391304j]
P(1): 0.6607


In [5]:
# ## 4. SamplerQNN과 EstimatorQNN의 출력
import numpy as np
from qiskit.circuit.library import z_feature_map, real_amplitudes
from qiskit.quantum_info import Statevector,SparsePauliOp
from qiskit_machine_learning.neural_networks import SamplerQNN,EstimatorQNN
from qiskit_machine_learning.primitives import QMLSampler,QMLEstimator
fmap=z_feature_map(2,reps=1)
ansatz=real_amplitudes(2,reps=1,entanglement='linear')
qc=fmap.compose(ansatz)
x=np.array([.2,.7])
theta=np.array([.1,.3,.5,.7])  # 세타: 네 학습 회전 각도
bindings=dict(zip(fmap.parameters,x))
bindings.update(dict(zip(ansatz.parameters,theta)))
state=Statevector.from_instruction(qc.assign_parameters(bindings))
sqnn=SamplerQNN(circuit=qc,sampler=QMLSampler(shots=None),input_params=list(fmap.parameters),
                weight_params=list(ansatz.parameters),interpret=lambda k:int(k).bit_count()%2,output_shape=2)
eqnn=EstimatorQNN(circuit=qc,estimator=QMLEstimator(default_precision=0),
                  input_params=list(fmap.parameters),weight_params=list(ansatz.parameters),
                  observables=SparsePauliOp('ZZ'),default_precision=0)
probability=np.asarray(sqnn.forward(x,theta)).ravel()
expectation=float(eqnn.forward(x,theta)[0,0])
assert np.isclose(probability.sum(),1)
assert np.isclose(probability[1],(1-expectation)/2)
print('비트열 00,01,10,11 확률:',np.round(state.probabilities(),6).tolist())
print('짝수/홀수 패리티 확률:',np.round(probability,6).tolist())
print('ZZ 기대값:',round(expectation,6))
score=-expectation+0.2
print('고전 출력층 P(1):',round(float(1/(1+np.exp(-score))),6))

비트열 00,01,10,11 확률: [0.286346, 0.163118, 0.137634, 0.412902]
짝수/홀수 패리티 확률: [0.699248, 0.300752]
ZZ 기대값: 0.398496
고전 출력층 P(1): 0.450538


In [6]:
# 5. 손실함수와 고전 최적화
import numpy as np
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
y=np.array([0,0,1,1])
p=np.array([.1,.6,.4,.8])
loss=-np.mean(y*np.log2(p)+(1-y)*np.log2(1-p))
pred=(p>=.5).astype(int)
print('교차엔트로피:',round(float(loss),6))
print('혼동행렬:',confusion_matrix(y,pred,labels=[0,1]).tolist())
print('정확도/정밀도/재현율/F1:',accuracy_score(y,pred),precision_score(y,pred),recall_score(y,pred),f1_score(y,pred))

교차엔트로피: 0.779447
혼동행렬: [[1, 1], [1, 1]]
정확도/정밀도/재현율/F1: 0.5 0.5 0.5 0.5
